# 국소 삭제·추가 압력으로 역확산 (L4)

1 → 2 → 3 실행. `sweep_...zip` 또는 이전 `fitted_reverse_...zip`을 업로드합니다.
Drive와 best.pt가 필요 없습니다. 이전 결과 ZIP에서는 **보정 전 원래 입력**을 다시 사용합니다.
candidate RGB 일치 손실은 사용하지 않습니다. 후보에서는 삭제 영역과 누락 중심선만 얻습니다.
첫 시험은 삭제만 활성화합니다. 2번에서 마스크를 확인하고 3번을 실행하세요.

빨강=삭제 / 파랑=추가. 정상적인 경계 차이는 허용하고, 나머지 영역에는 입력 보존 손실을 약하게 적용합니다.
확산 모델은 전체 이미지를 예측하며 가이드는 국소 latent에만 적용됩니다.
자동 마스크도 틀릴 수 있으므로 빨강이 정상 획을 넓게 덮지 않는지 확인하세요.
원하면 2번의 ENABLE_RESTORE=True로 중심선 추가도 켤 수 있습니다.
이번 실험은 기준 획 정보가 있는 조건이며, 정답 없이 보정 성능이 입증됐다는 뜻은 아닙니다.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'diffusers==0.37.0', 'transformers==4.57.6', 'accelerate',
    'safetensors', 'numpy', 'pillow', 'matplotlib', 'scipy', 'scikit-image'], check=True)
from pathlib import Path
import importlib.util
HELPER_SOURCE = '"""Paired DDIM re-denoising guided by an extractor-fitted candidate, not GT."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\n\ndef load_case(archive, case_id):\n    """Read named members only; recorded Colab absolute paths are not trusted."""\n    with zipfile.ZipFile(archive) as z:\n        if \'case_manifest.json\' not in z.namelist():\n            report = json.loads(z.read(\'report.json\'))\n            metadata = dict(report[\'metadata\'])\n            if metadata[\'case\'][\'case_id\'] != case_id:\n                raise ValueError(\'CASE_ID does not match the saved reverse experiment\')\n            with Image.open(io.BytesIO(z.read(\'input.png\'))) as im:\n                source = im.convert(\'RGB\').copy()\n            with Image.open(io.BytesIO(z.read(\'fitted_candidate.png\'))) as im:\n                target = im.convert(\'RGB\').copy()\n            if source.size != target.size or any(n % 8 for n in source.size):\n                raise ValueError(\'Input/candidate dimensions mismatch\')\n            metadata[\'reused_archive_kind\'] = \'reverse_result_original_input_and_candidate\'\n            return source, target, metadata\n        cases = json.loads(z.read(\'case_manifest.json\'))\n        matches = [c for c in cases if c[\'case_id\'] == case_id]\n        if len(matches) != 1:\n            raise ValueError(\'Select exactly one case from case_manifest.json\')\n        case = matches[0]\n        page = f"generation/page_{case[\'page_id\']:03d}.png"\n        candidate = f"{case_id}/normal_candidate.png"\n        def read_image(name):\n            with Image.open(io.BytesIO(z.read(name))) as im:\n                return im.convert(\'RGB\').copy()\n        source, target = read_image(page), read_image(candidate)\n        if source.size != target.size or any(n % 8 for n in source.size):\n            raise ValueError(\'Input and fitted candidate must have equal dimensions divisible by 8\')\n        report = json.loads(z.read(\'generation/generation_report.json\'))\n        metadata = dict(case=case, candidate_member=candidate, input_member=page,\n                        generation_model=report[\'model_id\'],\n                        candidate_is_ground_truth=False,\n                        prompt=report.get(\'prompts\', {}).get(\'4\', \'\'),\n                        negative_prompt=report.get(\'negative_prompt\', \'\'))\n    return source, target, metadata\n\n\ndef image_tensor(image, device):\n    a = np.asarray(image, dtype=np.float32) / 255\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].to(device)\n\n\nclass InkConstraintObjective:\n    """Candidate supplies fixed sparse constraints, never RGB target pixels."""\n    def __init__(self, source, candidate, *, enable_restore=False, tolerance=5.,\n                 halo=2, preserve_weight=.5, background_ink=.015, required_ink=.7):\n        from scipy.ndimage import distance_transform_edt, binary_dilation\n        from skimage.morphology import skeletonize\n        if source.shape != candidate.shape or source.shape[:2] != (1, 3):\n            raise ValueError(\'Expected matching single RGB sheets\')\n        if not np.isfinite(tolerance) or tolerance < 2 or type(halo) is not int or halo < 0:\n            raise ValueError(\'Invalid constraint tolerance/halo\')\n        if not 0 <= background_ink < required_ink <= 1 or preserve_weight < 0:\n            raise ValueError(\'Invalid ink constraints\')\n        self.source = source.detach()\n        self.preserve_weight = preserve_weight\n        self.background_ink, self.required_ink = background_ink, required_ink\n        source_ink = self.ink(source).detach().cpu().numpy()[0, 0]\n        target_ink = self.ink(candidate).detach().cpu().numpy()[0, 0]\n        source_binary, target_binary = source_ink > .35, target_ink > .35\n        if not source_binary.any() or not target_binary.any():\n            raise ValueError(\'Empty source or candidate: cannot infer safe correspondence\')\n        distance = distance_transform_edt(~target_binary)\n        # Ignore ordinary boundary/width differences within a tolerance corridor.\n        seeds = source_binary & (distance > tolerance)\n        nearby = binary_dilation(seeds, iterations=halo) if halo else seeds\n        erase = nearby*np.clip((distance-1)/max(tolerance-1, 1), 0, 1)\n        # Restore only an absent centerline, not the candidate\'s full silhouette.\n        missing = skeletonize(target_binary) & (distance_transform_edt(~source_binary) > tolerance)\n        restore = missing.astype(np.float32) if enable_restore else np.zeros_like(erase)\n        erase[restore > 0] = 0  # Never issue conflicting instructions.\n        def tensor(a):\n            return torch.as_tensor(a.copy(), dtype=torch.float32, device=source.device)[None, None]\n        self.erase, self.restore = tensor(erase), tensor(restore)\n        self.edit = torch.maximum(self.erase, self.restore)\n        occupied = (self.edit > 0).float()\n        self.preserve = 1-F.max_pool2d(occupied, 5, 1, 2)\n        self.metadata = dict(enable_restore=enable_restore, tolerance_pixels=tolerance,\n            halo_pixels=halo, preserve_weight=preserve_weight,\n            background_ink=background_ink, required_ink=required_ink,\n            erase_pixels=int((self.erase > 0).sum()), restore_pixels=int((self.restore > 0).sum()),\n            candidate_rgb_matching=False, masks_frozen_for_entire_pass=True)\n\n    @staticmethod\n    def ink(rgb):\n        return 1-(rgb*rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n\n    def project_gradient(self, gradient):\n        # A small latent context halo permits VAE reconstruction at edit boundaries.\n        support = F.adaptive_max_pool2d((self.edit > 0).float(), gradient.shape[-2:])\n        return gradient*F.max_pool2d(support, 3, 1, 1)\n\n    def __call__(self, image):\n        ink = self.ink(image)\n        remaining = F.relu(ink-self.background_ink)\n        erase = CandidateObjective.mean_region(remaining, self.erase)\n        # Penalize the darkest residual pixels as well as average remaining ink.\n        active = remaining[self.erase > 0]\n        tail = active.topk(max(1, int(active.numel()*.1))).values.mean() if active.numel() else erase*0\n        restore = CandidateObjective.mean_region(F.relu(self.required_ink-ink), self.restore)\n        preserve = CandidateObjective.mean_region((image-self.source).square(), self.preserve)\n        healthy_ink = self.preserve*(self.ink(self.source) > .2).float()\n        healthy_mse = CandidateObjective.mean_region((image-self.source).square(), healthy_ink)\n        loss = erase+.25*tail+restore+self.preserve_weight*preserve\n        return loss, dict(erase_ink=float(erase.detach()), erase_tail=float(tail.detach()),\n            restore_deficit=float(restore.detach()), preserved_region_mse=float(preserve.detach()),\n            healthy_ink_rgb_mse=float(healthy_mse.detach()))\n\n\nclass CandidateObjective:\n    """Fixed spatial weights; gradients stay continuous through the VAE."""\n    def __init__(self, source, candidate, threshold=.10, radius=2, preserve_weight=.25):\n        if source.shape != candidate.shape or not 0 < threshold < 1 or radius < 0:\n            raise ValueError(\'Invalid images or edit-region settings\')\n        self.source, self.candidate = source.detach(), candidate.detach()\n        difference = (source - candidate).abs().mean(1, keepdim=True)\n        edit = (difference > threshold).float()\n        self.edit = F.max_pool2d(edit, 2*radius+1, 1, radius).detach()\n        self.preserve_weight = preserve_weight\n\n    @staticmethod\n    def mean_region(values, mask):\n        return (values*mask).sum() / (mask.sum()*values.shape[1]).clamp_min(1)\n\n    def __call__(self, image):\n        match = self.mean_region((image-self.candidate).square(), self.edit)\n        preserve = self.mean_region((image-self.source).square(), 1-self.edit)\n        loss = match + self.preserve_weight*preserve\n        return loss, dict(candidate_region_mse=float(match.detach()),\n                          preserved_region_mse=float(preserve.detach()))\n\n\ndef optimize_clean(clean, decode, objective, *, iterations=4, step_ratio=.02,\n                   budget_ratio=.08, backtracks=5):\n    """Backtrack on the SAME clean estimate; bound total correction per step."""\n    anchor = clean.detach().float()\n    current = anchor.clone()\n    scale = anchor.square().mean().sqrt().clamp_min(1e-3)\n    with torch.no_grad():\n        before, _ = objective(decode(current))\n    accepted = 0\n    rejected_guards = 0\n    for _ in range(iterations):\n        with torch.enable_grad():\n            leaf = current.detach().requires_grad_(True)\n            loss, _ = objective(decode(leaf))\n            gradient, = torch.autograd.grad(loss, leaf)\n            if hasattr(objective, \'project_gradient\'):\n                gradient = objective.project_gradient(gradient)\n        if not torch.isfinite(gradient).all() or not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite candidate guidance\')\n        norm = gradient.square().mean().sqrt()\n        if norm < 1e-12:\n            break\n        direction = step_ratio*scale*gradient/norm\n        found = False\n        with torch.no_grad():\n            before_image = decode(current) if hasattr(objective, \'check_proposal\') else None\n            for attempt in range(backtracks+1):\n                proposal = current-direction*(.5**attempt)\n                delta = proposal-anchor\n                rms = delta.square().mean().sqrt().clamp_min(1e-12)\n                proposal = anchor+delta*(budget_ratio*scale/rms).clamp(max=1)\n                trial_image = decode(proposal)\n                trial, _ = objective(trial_image)\n                guard_ok = True\n                if torch.isfinite(trial) and trial < loss.detach() and hasattr(objective, \'check_proposal\'):\n                    guard_ok = objective.check_proposal(before_image, trial_image)\n                    rejected_guards += int(not guard_ok)\n                if torch.isfinite(trial) and trial < loss.detach() and guard_ok:\n                    current = proposal.detach()\n                    accepted += 1\n                    found = True\n                    break\n        if not found:\n            break\n    with torch.no_grad():\n        after, details = objective(decode(current))\n    return current, dict(loss_before=float(before), loss_after=float(after),\n                         accepted_inner_steps=accepted, rejected_guard_attempts=rejected_guards, **details)\n\n\ndef run_pair(source, candidate, metadata, output, *, steps=60, strength=.35,\n             seed=2026, cfg_scale=5., inner_steps=4, step_ratio=.02,\n             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n             progress=None, guidance_mode=\'candidate\', constraint_config=None, objective_override=None):\n    """Identical source VAE encoding, noise, prompt, scheduler in both arms."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select an L4 GPU runtime in Colab\')\n    if not 0 < strength <= 1 or steps < 4 or int(steps*strength) < 2:\n        raise ValueError(\'Need at least two actual re-denoising steps\')\n    if inner_steps < 1 or step_ratio <= 0 or cfg_scale <= 1:\n        raise ValueError(\'Invalid guidance configuration\')\n    if guidance_mode not in (\'candidate\', \'ink_constraints\', \'adaptive_ink\'):\n        raise ValueError(\'Unknown guidance mode\')\n    if (guidance_mode == \'adaptive_ink\') != (objective_override is not None):\n        raise ValueError(\'Adaptive mode requires an explicitly prepared stroke objective\')\n    from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n    from copy import deepcopy\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    source.save(out/\'input.png\')\n    candidate.save(out/\'fitted_candidate.png\')\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    if revision:\n        kwargs[\'revision\'] = revision\n    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config,\n                                              clip_sample=False, thresholding=False)\n    pipe.to(\'cuda\')\n    pipe.enable_attention_slicing()\n    pipe.enable_vae_slicing()\n    for module in (pipe.unet, pipe.vae, pipe.text_encoder):\n        module.eval().requires_grad_(False)\n    prompt = metadata.get(\'prompt\') or (\'four Korean Hangul glyphs, black ink on white, \'\n                                      \'preserve the input typography and layout\')\n    with torch.no_grad():\n        positive, negative = pipe.encode_prompt(prompt, \'cuda\', 1, True,\n            negative_prompt=metadata.get(\'negative_prompt\', \'\'))\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        # VAE and noise initialization exactly once, then copy for both arms.\n        initial = pipe.prepare_latents(pipe.image_processor.preprocess(source), timesteps[:1],\n            1, 1, embeddings.dtype, \'cuda\', torch.Generator(\'cpu\').manual_seed(seed))\n    pipe.vae.to(dtype=torch.float32)\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,\n                               return_dict=False)[0].float()/2+.5\n    def save_image(z, path):\n        with torch.no_grad():\n            array = decode(z)[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(array*255).astype(\'uint8\')).save(path)\n    source_rgb, candidate_rgb = image_tensor(source, \'cuda\'), image_tensor(candidate, \'cuda\')\n    if objective_override is not None and (objective_override.source.shape != source_rgb.shape or\n            not torch.allclose(objective_override.source, source_rgb, atol=1e-6)):\n        raise ValueError(\'Adaptive objective must be prepared from this exact original input\')\n    objective = objective_override if objective_override is not None else (InkConstraintObjective(source_rgb, candidate_rgb, **(constraint_config or {}))\n                 if guidance_mode == \'ink_constraints\' else CandidateObjective(source_rgb, candidate_rgb))\n    Image.fromarray((objective.edit[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/\'edit_region.png\')\n    if guidance_mode in (\'ink_constraints\', \'adaptive_ink\'):\n        for name in (\'erase\', \'restore\', \'preserve\'):\n            Image.fromarray(np.rint(getattr(objective, name)[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/f\'{name}_region.png\')\n        (out/\'constraints.json\').write_text(json.dumps(objective.metadata, indent=2), encoding=\'utf-8\')\n    guided_name = {\'candidate\': \'candidate_guided\', \'ink_constraints\': \'ink_guided\', \'adaptive_ink\': \'adaptive_guided\'}[guidance_mode]\n    if hasattr(objective, \'export_state\'):\n        objective.export_state(out/\'ownership_initial.npz\')\n    traces, metrics = {}, {}\n    for method in (\'redenoise_only\', guided_name):\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.detach().float().clone()\n        trace = []\n        for i, t in enumerate(timesteps):\n            with torch.no_grad():\n                inputs = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(inputs.to(torch.float16), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                noise = (u+cfg_scale*(c-u)).float()\n                base = scheduler.step(noise, t, latent, eta=0., return_dict=True)\n            record = dict(step=i, timestep=int(t), active=False, accepted_inner_steps=0)\n            latent = base.prev_sample.detach()\n            clean = base.pred_original_sample.detach()\n            if method == \'redenoise_only\' and hasattr(objective, \'record_baseline\'):\n                with torch.no_grad():objective.record_baseline(i, decode(clean))\n            if method == guided_name and i/max(len(timesteps)-1, 1) >= .2:\n                if hasattr(objective, \'begin_step\'):objective.begin_step(i)\n                if hasattr(objective, \'maybe_refresh\'):\n                    with torch.no_grad():\n                        refresh = objective.maybe_refresh(decode(clean), i)\n                    if refresh is not None:\n                        record[\'constraint_refresh\'] = refresh\n                        for name in (\'erase\', \'restore\'):\n                            mask = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                            Image.fromarray(np.rint(mask*255).clip(0, 255).astype(\'uint8\')).save(out/f\'{name}_step_{i:03d}.png\')\n                        if hasattr(objective, \'export_state\'):\n                            objective.export_state(out/f\'ownership_step_{i:03d}.npz\')\n                clean, result = optimize_clean(clean, decode, objective,\n                    iterations=inner_steps if i < len(timesteps)-1 else inner_steps*2,\n                    step_ratio=step_ratio)\n                # DDIM eta=0: x_(t-1) = sqrt(a_prev)*x0 + sqrt(1-a_prev)*epsilon.\n                # Hold the predicted epsilon fixed and change the clean component.\n                prev_t = int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[prev_t] if prev_t >= 0\n                         else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent+alpha.sqrt()*(clean-base.pred_original_sample)\n                record.update(active=True, **result)\n            latent = latent.detach()\n            trace.append(record)\n            if i % 4 == 0 or i == len(timesteps)-1:\n                save_image(clean, out/f\'{method}_step_{i:03d}.png\')\n            if progress:\n                progress(method, i+1, len(timesteps), record)\n        save_image(latent, out/f\'{method}.png\')\n        with torch.no_grad():\n            evaluate = getattr(objective, \'evaluate_fixed\', objective)\n            loss, values = evaluate(decode(latent))\n        metrics[method] = dict(loss=float(loss), **values)\n        if hasattr(objective, \'finish_arm\'):\n            with torch.no_grad():objective.finish_arm(method, decode(latent))\n        traces[method] = trace\n    with torch.no_grad():\n        loss, values = getattr(objective, \'evaluate_fixed\', objective)(objective.source)\n    metrics[\'input\'] = dict(loss=float(loss), **values)\n    report = dict(metadata=metadata, model_id=model_id, revision=revision,\n        steps=steps, actual_steps=len(timesteps), strength=strength, seed=seed,\n        cfg_scale=cfg_scale, inner_steps=inner_steps, step_ratio=step_ratio,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True, guidance_target=guidance_mode,\n        constraints=getattr(objective, \'metadata\', None),\n        constraint_history=getattr(objective, \'history\', None),\n        target_pixels_pasted_into_output=False, metrics=metrics, traces=traces,\n        metric_note=(\'Fixed ink constraints and source preservation; not GT accuracy or HSQM score\'\n                     if guidance_mode in (\'ink_constraints\', \'adaptive_ink\') else \'Candidate agreement, not ground-truth accuracy or HSQM score\'))\n    if hasattr(objective, \'select_result\'):\n        import shutil\n        selection = objective.select_result(guided_name)\n        report[\'selection\'] = selection\n        shutil.copy2(out/selection[\'file\'], out/\'selected.png\')\n    (out/\'report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    del pipe\n    torch.cuda.empty_cache()\n    return report\n'
helper_path=Path('/content/fitted_candidate_diffusion_runtime.py')
helper_path.write_text(HELPER_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('fitted_candidate_diffusion_runtime',helper_path)
repair=importlib.util.module_from_spec(spec);spec.loader.exec_module(repair)
import torch
if not torch.cuda.is_available():raise RuntimeError('런타임 → 런타임 유형 변경 → L4 GPU를 선택하세요.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
import json, zipfile
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np

ZIP_PATH=''
CASE_ID='case_003_strength_0p65_seed_100'
ENABLE_RESTORE=False  # 첫 실행: 삭제만. 누락 중심선 추가 시험은 True.
TOLERANCE=5.          # 현재 576px 이미지 기준. 정상 윤곽 차이를 허용하는 거리.
CONSTRAINT_CONFIG=dict(enable_restore=ENABLE_RESTORE,tolerance=TOLERANCE,
                       halo=2,preserve_weight=.5,background_ink=.015,required_ink=.7)
if not ZIP_PATH:
    uploaded=files.upload()
    names=[name for name in uploaded if name.lower().endswith('.zip')]
    if len(names)!=1:raise ValueError('결과 ZIP 하나를 선택하세요.')
    ZIP_PATH=names[0]
with zipfile.ZipFile(ZIP_PATH) as z:
    available=(json.loads(z.read('case_manifest.json')) if 'case_manifest.json' in z.namelist()
               else [json.loads(z.read('report.json'))['metadata']['case']])
print('사용 가능한 사례:',[c['case_id'] for c in available])
source,candidate,metadata=repair.load_case(ZIP_PATH,CASE_ID)
preview=repair.InkConstraintObjective(repair.image_tensor(source,'cpu'),
    repair.image_tensor(candidate,'cpu'),**CONSTRAINT_CONFIG)
erase=preview.erase[0,0].numpy();restore=preview.restore[0,0].numpy()
overlay=np.asarray(source,dtype=np.float32)/255
alpha=.7*np.maximum(erase,restore)[...,None]
color=np.stack([erase,np.zeros_like(erase),restore],-1)
color=color/np.maximum(color.max(-1,keepdims=True),1e-8)
overlay=overlay*(1-alpha)+color*alpha
fig,axes=plt.subplots(1,4,figsize=(20,5))
for ax,im,title in zip(axes,[source,overlay,erase,restore],
        ['Original erroneous input','Red=erase / blue=restore','Erase pressure','Restore centerline']):
    ax.imshow(im,cmap='magma',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.show();plt.close(fig)
print('제약:',preview.metadata)
print('자동 마스크가 잘못됐으면 3번을 실행하지 말고 TOLERANCE를 조정하세요.')

In [ ]:
from datetime import datetime
import shutil
from PIL import Image
from huggingface_hub import model_info

# .35는 이번 추가 역확산의 강도입니다. 원래 오류 생성 강도 .65와 별개입니다.
STRENGTH=.25
STEPS=60  # 실제 추가 역확산은 int(STEPS*STRENGTH)=15단계
INNER_STEPS=4
STEP_RATIO=.012
SEED=2026
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION=model_info(MODEL_ID).sha
OUTPUT=Path('/content')/('ink_constraint_reverse_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
def progress(method,step,total,record):
    if step==1 or step%4==0 or step==total:
        print(f"{method}: {step}/{total} | accepted={record['accepted_inner_steps']} | "
              f"loss={record.get('loss_before')} → {record.get('loss_after')}",flush=True)
report=repair.run_pair(source,candidate,metadata,OUTPUT,steps=STEPS,strength=STRENGTH,
    seed=SEED,inner_steps=INNER_STEPS,step_ratio=STEP_RATIO,
    model_id=MODEL_ID,revision=REVISION,progress=progress,guidance_mode='ink_constraints',constraint_config=CONSTRAINT_CONFIG)
panels=[('Input',source),('Re-denoise only',Image.open(OUTPUT/'redenoise_only.png')),
        ('Sparse ink-guided reverse diffusion',Image.open(OUTPUT/'ink_guided.png')),
        ('Candidate (not an RGB target)',candidate)]
fig,axes=plt.subplots(1,4,figsize=(20,5))
for ax,(title,im) in zip(axes,panels):
    ax.imshow(im);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'comparison.png',dpi=140);plt.show();plt.close(fig)
print('실제 추가 단계:',report['actual_steps'])
print('잉크 제약과 원본 보존 손실(정확도 아님):',json.dumps(report['metrics'],indent=2))
for method in ('redenoise_only','ink_guided'):
    snapshots=sorted(OUTPUT.glob(method+'_step_*.png'))
    fig,axes=plt.subplots(1,len(snapshots),figsize=(4*len(snapshots),4),squeeze=False)
    for ax,path in zip(axes.flat,snapshots):
        ax.imshow(Image.open(path));ax.set_title(path.stem);ax.axis('off')
    fig.tight_layout();fig.savefig(OUTPUT/(method+'_trajectory.png'),dpi=120);plt.show();plt.close(fig)
(OUTPUT/'runtime_source.py').write_text(HELPER_SOURCE,encoding='utf-8')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('결과:',archive)
files.download(archive)